In [1]:
import pandas as pd
import numpy as np
import os

raw_path = "../data/raw"
intern_df = pd.read_excel(os.path.join(raw_path, "Haryana_Government_Internships_Dataset.xlsx")) 
intern_df.shape, intern_df.head()

((1000, 15),
        Course_Name                Internship_Role  \
 0       ITI Welder   Police Administration Intern   
 1  ITI Electrician          Urban Planning Intern   
 2     B.Tech Civil  Electrical Maintenance Intern   
 3  ITI Electrician              Civil Site Intern   
 4       B.Pharmacy    Software Development Intern   
 
                  Government_Department              Sector Location_City  \
 0         Haryana Transport Department  Finance & Accounts         Sirsa   
 1  Haryana Renewable Energy Department           Education       Panipat   
 2         Haryana Education Department           Education        Ambala   
 3                Haryana IT Department              Energy       Panipat   
 4      Haryana Public Works Department          Healthcare        Ambala   
 
    Minimum_Age  Maximum_Age  Duration_Months  Stipend_Per_Month_INR     Mode  \
 0           18           28                2                      0   Hybrid   
 1           18           29       

In [2]:
intern_df.columns = (
    intern_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)
intern_df.columns

Index(['course_name', 'internship_role', 'government_department', 'sector',
       'location_city', 'minimum_age', 'maximum_age', 'duration_months',
       'stipend_per_month_inr', 'mode', 'minimum_percentage_required',
       'skill_requirement', 'certificate_provided',
       'pre_placement_offer_possibility', 'application_mode'],
      dtype='object')

In [3]:
text_cols = [
    'course_name','internship_role','government_department','sector',
    'location_city','mode','skill_requirement','certificate_provided',
    'pre_placement_offer_possibility','application_mode'
]

for c in text_cols:
    intern_df[c] = intern_df[c].astype(str).str.strip()

# lower-case for consistent matching
for c in ['sector','location_city','mode','certificate_provided','pre_placement_offer_possibility','application_mode']:
    intern_df[c] = intern_df[c].str.lower()

In [4]:
num_cols = [
    'minimum_age','maximum_age','duration_months',
    'stipend_per_month_inr','minimum_percentage_required'
]

for c in num_cols:
    intern_df[c] = pd.to_numeric(intern_df[c], errors='coerce')

In [5]:
intern_df[num_cols].isnull().sum()

minimum_age                    0
maximum_age                    0
duration_months                0
stipend_per_month_inr          0
minimum_percentage_required    0
dtype: int64

In [6]:
# invalid age ranges
invalid_age = (intern_df['minimum_age'] > intern_df['maximum_age']).sum()
print("Invalid age rows:", invalid_age)

# negative values
neg_stipend = (intern_df['stipend_per_month_inr'] < 0).sum()
print("Negative stipend rows:", neg_stipend)

Invalid age rows: 0
Negative stipend rows: 0


In [7]:
yes_map = {
    'yes':'yes','y':'yes','true':'yes','1':'yes',
    'no':'no','n':'no','false':'no','0':'no'
}

intern_df['certificate_provided'] = intern_df['certificate_provided'].replace(yes_map)
intern_df['pre_placement_offer_possibility'] = intern_df['pre_placement_offer_possibility'].replace(yes_map)

In [8]:
intern_df['stipend_bucket'] = pd.cut(
    intern_df['stipend_per_month_inr'],
    bins=[-1, 0, 5000, 10000, 20000, 1000000],
    labels=['unpaid','low','medium','good','high']
)

intern_df['duration_bucket'] = pd.cut(
    intern_df['duration_months'],
    bins=[0,1,3,6,12,60],
    labels=['very_short','short','medium','long','very_long']
)

In [9]:
intern_df.to_csv("../data/cleaned/internships_cleaned.csv", index=False)